# Hai nhánh độc lập: bỏ hòa phiếu và giữ hòa phiếu
Mỗi nhánh lọc dữ liệu, xử lý hòa phiếu và khử trùng xong rồi mới chia stratify 70–15–15 (làm tròn số ảnh).
Cell 1 chuẩn bị hai bộ chia; cell 2 định nghĩa huấn luyện; cell 3 train lần lượt hai nhánh với cùng ba seed;
cell 4 đánh giá test riêng và Grad-CAM; cell 5 kiểm định McNemar trên giao của hai test.
Loss giữ phân phối phiếu mềm. Với ảnh hòa, nhãn tham chiếu để stratify/chấm điểm được chọn có seed giữa các lớp đồng hạng.
Không nạp checkpoint cũ. Mỗi nhánh chọn checkpoint và hiệu chỉnh bằng validation riêng, không chọn bằng test.
Phần giao chỉ gồm ảnh không hòa phiếu, không bảo đảm tỷ lệ lớp; xuất số ảnh và phân bố lớp cùng kết quả kiểm định.
Kết quả các lần train trước không phải kết quả của notebook này.


In [ ]:
# =====================================================================
# CELL 1: NẠP DỮ LIỆU, KHỬ TRÙNG 2 TẦNG (SHA-256 + dHASH), TRỌNG SỐ ĐÁM ĐÔNG & CHIA STRATIFIED 70/15/15
# =====================================================================
import importlib
import os
import random
import sys
from pathlib import Path
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader, WeightedRandomSampler


def set_seed(seed=42):
    """Cố định hạt giống ngẫu nhiên (seed) để đảm bảo tính tái lập kết quả."""
    random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


KAGGLE_INPUT = Path("/kaggle/input")


def find_unique_file(filename):
    if Path(filename).exists():
        return Path(filename).resolve()
    matches = list(KAGGLE_INPUT.rglob(filename)) if KAGGLE_INPUT.exists() else []
    if len(matches) != 1:
        paths = "\n".join(str(path) for path in matches) or "(không tìm thấy)"
        raise FileNotFoundError(
            f"Cần tìm thấy đúng 1 file '{filename}', nhưng tìm thấy {len(matches)}:\n{paths}"
        )
    return matches[0]


DATA_UTILS = find_unique_file("data_utils.py")
PROJECT_DIR = DATA_UTILS.parent
IMG_CSV = find_unique_file("fer2013.csv")
LABEL_CSV = find_unique_file("fer2013new.csv")
# Both branches use the same training settings, but independent stratified splits.
import json
import pandas as pd
import gc
BRANCHES = ("drop_ties", "keep_ties")
SEEDS = [42, 100, 2026]
SPLIT_SEED = 42
EVAL_PROTOCOL = "single"  # Fixed before training; identical for both branches.
OUTPUT_ROOT = Path("/kaggle/working") if KAGGLE_INPUT.exists() else Path.cwd() / "outputs"
EXPERIMENT_ROOT = (OUTPUT_ROOT / "two_branches_stratified_70_15_15").resolve()
EXPERIMENT_ROOT.mkdir(parents=True, exist_ok=True)

if str(PROJECT_DIR) not in sys.path:
    sys.path.insert(0, str(PROJECT_DIR))

print("Thư mục mã nguồn:", PROJECT_DIR)
print("File ảnh:", IMG_CSV)
print("File nhãn:", LABEL_CSV)

from comparison_utils import export_predictions, compare_test_intersection
import preprocessing
import data_utils
import dataset
import models
import eval_utils

importlib.reload(preprocessing)
importlib.reload(data_utils)
importlib.reload(dataset)
importlib.reload(models)
importlib.reload(eval_utils)

from preprocessing import PREPROCESSING_CONFIG, preprocess_fer_gray_48x48
from data_utils import load_fer2013_csv, load_split_from_manifest, compute_file_sha256, LABEL_NAMES
from dataset import FER2013Dataset, mixup_soft_batch
from models import CustomCNN
from eval_utils import (
    SoftWeightedKLDivLoss,
    ModelEMA,
    get_cosine_gamma,
    compute_smooth_class_weights,
    compute_effective_number_weights,
    compute_class_priors,
    apply_posthoc_logit_adjustment,
    optimize_classwise_posthoc_tau,
    compute_epoch_metrics,
    summarize_multi_seed_results,
    plot_learning_curves,
    denormalize_image,
    ten_crop_inference,
    generate_gradcam,
    evaluate_private_test,
)

# Finish filtering/tie handling/deduplication independently BEFORE each split.
# No samples are appended to any split afterwards.
for branch in BRANCHES:
    branch_dir = EXPERIMENT_ROOT / branch
    branch_dir.mkdir(parents=True, exist_ok=True)
    prepared = load_fer2013_csv(
        IMG_CSV, LABEL_CSV, return_soft_labels=True, return_sample_weights=True,
        use_stratify=True, use_multilabel_stratify=False,
        use_perceptual_dedup=True, near_dup_hamming=0, near_dup_pixel_mad_threshold=15.0,
        drop_ties=(branch == "drop_ties"), retain_rare_ties_in_train=False,
        tie_reference_policy="seeded", val_size=0.15, test_size=0.15,
        random_state=SPLIT_SEED, split_manifest_dir=branch_dir / "split_manifest",
        allow_positional_join=True,
    )
    del prepared
    gc.collect()
    (branch_dir / "experiment_config.json").write_text(json.dumps({
        "experiment": branch, "evaluation_protocol": EVAL_PROTOCOL, "seeds": SEEDS,
        "split_policy": "clean_all_then_independent_stratified_holdout",
        "split_ratios": [0.70, 0.15, 0.15], "split_seed": SPLIT_SEED,
        "tie_reference_policy": "seeded", "loss_target": "soft_vote_distribution",
        "metric_target": "fixed manifest reference label",
        "mcnemar": "intersection of independently held-out tests only",
    }, indent=2), encoding="utf-8")


def activate_branch(branch):
    # Reconstruct saved splits; never reshuffle when training/evaluating.
    global EXPERIMENT, RUN_DIR, SPLIT_MANIFEST_DIR
    global X_train, y_train_soft, y_train_hard, w_train
    global X_val, y_val_soft, y_val_hard, w_val
    global X_test, y_test_soft, y_test_hard, w_test
    global class_priors, smooth_class_weights
    if branch not in BRANCHES:
        raise ValueError(branch)
    EXPERIMENT = branch
    RUN_DIR = EXPERIMENT_ROOT / branch
    SPLIT_MANIFEST_DIR = RUN_DIR / "split_manifest"
    ((X_train, y_train_soft, y_train_hard, w_train),
     (X_val, y_val_soft, y_val_hard, w_val),
     (X_test, y_test_soft, y_test_hard, w_test)) = load_split_from_manifest(
         IMG_CSV, LABEL_CSV, SPLIT_MANIFEST_DIR, allow_positional_join=True,
         verify_source_hash=True)
    clean_class_counts = y_train_soft.sum(axis=0)
    class_priors = compute_class_priors(clean_class_counts)
    smooth_class_weights = compute_smooth_class_weights(clean_class_counts, power=0.25, max_ratio=3.0)
    os.chdir(RUN_DIR)
    print(f"{branch}: train={len(X_train)}, val={len(X_val)}, test={len(X_test)}")

EMOTION_DICT = {
    0: "Neutral",
    1: "Happiness",
    2: "Surprise",
    3: "Sadness",
    4: "Anger",
    5: "Disgust",
    6: "Fear",
    7: "Contempt",
}

In [ ]:
# =====================================================================
# CELL 2: HUẤN LUYỆN CUSTOM CNN (PARALLEL ECA+SE + RESIDUAL SPATIAL ATTENTION) + BALANCED LOGIT-ADJ
# =====================================================================
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Đang huấn luyện trên thiết bị:", device)

# Priors and weights are computed separately by activate_branch().

def create_dataloaders(
    seed,
    batch_size=64,
    oversample_alpha=0.5,
    sampler_use_sample_weight=True,
):
    """
    oversample_alpha in [0.0, 1.0]:
      - 0.0: lấy mẫu ngẫu nhiên thuần tuý.
      - >0.0: oversampling nhẹ cho các lớp hiếm kết hợp trọng số mẫu.
    sampler_use_sample_weight (Mặc định True = hành vi hiện tại):
      - Khi True: sampler_weights = class_boost * w_train.
      - Khi False: sampler_weights = class_boost (tránh nhân sample_weight hai lần ở cả sampler và loss).
    """
    set_seed(seed)
    train_dataset = FER2013Dataset(
        X_train,
        y_train_soft,
        is_train=True,
        erasing_prob=0.15,
        erasing_scale=(0.01, 0.04),
        sample_weights=w_train,
        metric_labels=y_train_hard,
        rare_classes=(5, 6, 7),
    )
    val_dataset = FER2013Dataset(X_val, y_val_soft, is_train=False, metric_labels=y_val_hard)
    test_dataset = FER2013Dataset(X_test, y_test_hard, is_train=False)

    g = torch.Generator()
    g.manual_seed(seed)

    print(
        f"[DataLoader | Seed {seed}] oversample_alpha={oversample_alpha}, "
        f"sampler_use_sample_weight={sampler_use_sample_weight}"
    )

    if oversample_alpha > 0.0:
        class_boost = (1.0 - oversample_alpha) + oversample_alpha * (y_train_soft @ smooth_class_weights)
        if sampler_use_sample_weight:
            sampler_weights = class_boost * w_train
        else:
            sampler_weights = class_boost
        sampler = WeightedRandomSampler(
            weights=torch.DoubleTensor(sampler_weights),
            num_samples=len(sampler_weights),
            replacement=True,
            generator=g,
        )
        train_loader = DataLoader(
            train_dataset,
            batch_size=batch_size,
            sampler=sampler,
            drop_last=True,
        )
    else:
        train_loader = DataLoader(
            train_dataset,
            batch_size=batch_size,
            shuffle=True,
            generator=g,
            drop_last=True,
        )

    val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)
    test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)
    return train_loader, val_loader, test_loader


def train_and_evaluate_custom_cnn(
    seed,
    num_epochs=85,
    ft_epochs=15,
    patience=25,
    mixup_alpha=0.2,
    mixup_prob=0.4,
    stage1_tau=0.30,
    stage1_max_gamma=0.5,
    gamma_warmup_epochs=15,
    posthoc_tau=0.10,
    use_ema=True,
    alpha_composite=0.5,
    sampler_use_sample_weight=True,
    log_val_before_after_tau=False,
):
    set_seed(seed)
    train_loader_p1, val_loader, test_loader = create_dataloaders(
        seed,
        batch_size=64,
        oversample_alpha=0.45,
        sampler_use_sample_weight=sampler_use_sample_weight,
    )

    model = CustomCNN(num_classes=8).to(device)
    model.freeze_backbone(False)
    ema = ModelEMA(model, decay=0.996) if use_ema else None

    criterion_p1 = SoftWeightedKLDivLoss(
        class_weights=None,
        class_priors=class_priors,
        logit_adj_tau=stage1_tau,
        gamma=0.0,
    ).to(device)

    optimizer = optim.SGD(
        model.parameters(), lr=0.01, momentum=0.9, weight_decay=1e-4, nesterov=True
    )
    warmup_epochs = min(5, max(1, num_epochs // 10))
    scheduler_warmup = optim.lr_scheduler.LinearLR(
        optimizer, start_factor=0.1, end_factor=1.0, total_iters=warmup_epochs
    )
    scheduler_cosine = optim.lr_scheduler.CosineAnnealingLR(
        optimizer, T_max=max(1, num_epochs - warmup_epochs), eta_min=1e-4
    )
    scheduler = optim.lr_scheduler.SequentialLR(
        optimizer,
        schedulers=[scheduler_warmup, scheduler_cosine],
        milestones=[warmup_epochs],
    )

    ckpt_path = f"best_custom_cnn_seed{seed}.pth"
    best_val_composite = 0.0
    best_val_macro_f1 = 0.0
    patience_counter = 0

    history = {
        "train_loss": [], "val_loss": [],
        "train_acc": [], "val_acc": [],
        "train_macro_f1": [], "val_macro_f1": [],
        "val_weighted_f1": [], "val_composite": [],
    }

    print(
        f"\n>>> [CustomCNN | Seed {seed}] GIAI ĐOẠN 1: HỌC BIỂU DIỄN "
        f"({num_epochs} Epochs, Mixup={mixup_alpha}@p={mixup_prob}, stage1_tau={stage1_tau}, "
        f"val_posthoc_tau={posthoc_tau}, max_gamma={stage1_max_gamma})..."
    )
    for epoch in range(num_epochs):
        if hasattr(train_loader_p1.dataset, "set_erasing_enabled"):
            train_loader_p1.dataset.set_erasing_enabled(epoch < (num_epochs - 15))

        cur_gamma = get_cosine_gamma(
            epoch, num_epochs, max_gamma=stage1_max_gamma, warmup_epochs=gamma_warmup_epochs
        )
        criterion_p1.set_gamma(cur_gamma)

        model.train()
        running_loss, total_train = 0.0, 0
        train_preds, train_targets = [], []

        for batch in train_loader_p1:
            images, soft_labels, sample_weights = (
                batch[0].to(device),
                batch[1].to(device),
                batch[2].to(device),
            )
            mixed_images, mixed_labels, mixed_weights = mixup_soft_batch(
                images,
                soft_labels,
                sample_weights=sample_weights,
                alpha=mixup_alpha,
                p=mixup_prob,
                rare_classes=(5, 6, 7),
            )

            optimizer.zero_grad()
            outputs = model(mixed_images)
            loss = criterion_p1(outputs, mixed_labels, sample_weights=mixed_weights)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=5.0)
            optimizer.step()

            if ema is not None:
                ema.update(model)

            bs = images.size(0)
            running_loss += loss.item() * bs
            total_train += bs
            train_preds.extend(outputs.argmax(dim=1).cpu().numpy())
            train_targets.extend(batch[3].cpu().numpy())

        train_loss = running_loss / total_train
        train_acc, train_macro_f1, _, _ = compute_epoch_metrics(train_targets, train_preds, alpha_composite)

        eval_net = ema.ema_model if ema is not None else model
        eval_net.eval()
        val_running_loss, total_val = 0.0, 0
        val_preds, val_targets = [], []

        with torch.no_grad():
            for images, soft_labels, reference_labels in val_loader:
                images, soft_labels = images.to(device), soft_labels.to(device)
                logits = eval_net(images)
                v_loss = criterion_p1(logits, soft_labels)
                adj_logits = apply_posthoc_logit_adjustment(logits, class_priors, tau=posthoc_tau)

                bs = images.size(0)
                val_running_loss += v_loss.item() * bs
                total_val += bs
                val_preds.extend(adj_logits.argmax(dim=1).cpu().numpy())
                val_targets.extend(reference_labels.cpu().numpy())

        val_loss = val_running_loss / total_val
        val_acc, val_macro_f1, val_weighted_f1, val_composite = compute_epoch_metrics(
            val_targets, val_preds, alpha_composite
        )

        history["train_loss"].append(train_loss)
        history["val_loss"].append(val_loss)
        history["train_acc"].append(train_acc)
        history["val_acc"].append(val_acc)
        history["train_macro_f1"].append(train_macro_f1)
        history["val_macro_f1"].append(val_macro_f1)
        history["val_weighted_f1"].append(val_weighted_f1)
        history["val_composite"].append(val_composite)

        current_lr = optimizer.param_groups[0]["lr"]
        scheduler.step()

        if val_composite > best_val_composite:
            best_val_composite = val_composite
            best_val_macro_f1 = val_macro_f1
            torch.save(eval_net.state_dict(), ckpt_path)
            patience_counter = 0
            flag = "-> Lưu best!"
        else:
            patience_counter += 1
            flag = ""

        if (epoch + 1) % 5 == 0 or flag:
            print(
                f"Epoch [{epoch+1:02d}/{num_epochs}] | LR: {current_lr:.5f} | Gamma: {cur_gamma:.2f} | "
                f"Loss: {train_loss:.4f}/{val_loss:.4f} | "
                f"Val Macro-F1: {val_macro_f1*100:.2f}% | Val Acc: {val_acc*100:.2f}% | "
                f"Val Comp: {val_composite*100:.2f}% {flag}"
            )

        if patience_counter >= patience:
            print(f"Dừng sớm (Early Stopping) Giai đoạn 1 tại Epoch {epoch + 1}!")
            break

    phase1_ran = len(history["train_loss"])

    # --- GIAI ĐOẠN 2: BLOCK4 + CLASSIFIER FINE-TUNING VỚI SMOOTH CLASS WEIGHTS ---
    print(f">>> [CustomCNN | Seed {seed}] GIAI ĐOẠN 2: BLOCK4 + CLASSIFIER FINE-TUNING ({ft_epochs} Epochs)...")
    best_state = torch.load(ckpt_path, map_location=device, weights_only=True)
    model.load_state_dict(best_state)
    model.freeze_backbone(True, unfreeze_block4=True)

    train_loader_p2, _, _ = create_dataloaders(
        seed,
        batch_size=64,
        oversample_alpha=0.0,
        sampler_use_sample_weight=sampler_use_sample_weight,
    )
    if hasattr(train_loader_p2.dataset, "set_erasing_enabled"):
        train_loader_p2.dataset.set_erasing_enabled(False)

    if ema is not None:
        ema = ModelEMA(model, decay=0.99)

    criterion_p2 = SoftWeightedKLDivLoss(
        class_weights=smooth_class_weights,
        class_priors=class_priors,
        logit_adj_tau=stage1_tau,
        gamma=0.0,
    ).to(device)

    optimizer_ft = optim.SGD(
        [
            {"params": model.block4.parameters(), "lr": 3e-4},
            {"params": model.classifier.parameters(), "lr": 1e-3},
        ],
        momentum=0.9,
        weight_decay=1e-4,
        nesterov=True,
    )
    scheduler_ft = optim.lr_scheduler.CosineAnnealingLR(optimizer_ft, T_max=ft_epochs, eta_min=1e-5)

    for epoch in range(ft_epochs):
        model.train()
        running_loss, total_train = 0.0, 0
        train_preds, train_targets = [], []

        for batch in train_loader_p2:
            images, soft_labels, sample_weights = (
                batch[0].to(device),
                batch[1].to(device),
                batch[2].to(device),
            )
            optimizer_ft.zero_grad()
            outputs = model(images)
            loss = criterion_p2(outputs, soft_labels, sample_weights=sample_weights)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=5.0)
            optimizer_ft.step()

            if ema is not None:
                ema.update(model)

            bs = images.size(0)
            running_loss += loss.item() * bs
            total_train += bs
            train_preds.extend(outputs.argmax(dim=1).cpu().numpy())
            train_targets.extend(batch[3].cpu().numpy())

        train_loss = running_loss / total_train
        train_acc, train_macro_f1, _, _ = compute_epoch_metrics(train_targets, train_preds, alpha_composite)

        eval_net = ema.ema_model if ema is not None else model
        eval_net.eval()
        val_running_loss, total_val = 0.0, 0
        val_preds, val_targets = [], []
        with torch.no_grad():
            for images, soft_labels, reference_labels in val_loader:
                images, soft_labels = images.to(device), soft_labels.to(device)
                logits = eval_net(images)
                v_loss = criterion_p2(logits, soft_labels)
                adj_logits = apply_posthoc_logit_adjustment(logits, class_priors, tau=posthoc_tau)

                bs = images.size(0)
                val_running_loss += v_loss.item() * bs
                total_val += bs
                val_preds.extend(adj_logits.argmax(dim=1).cpu().numpy())
                val_targets.extend(reference_labels.cpu().numpy())

        val_loss = val_running_loss / total_val
        val_acc, val_macro_f1, val_weighted_f1, val_composite = compute_epoch_metrics(
            val_targets, val_preds, alpha_composite
        )
        scheduler_ft.step()

        history["train_loss"].append(train_loss)
        history["val_loss"].append(val_loss)
        history["train_acc"].append(train_acc)
        history["val_acc"].append(val_acc)
        history["train_macro_f1"].append(train_macro_f1)
        history["val_macro_f1"].append(val_macro_f1)
        history["val_weighted_f1"].append(val_weighted_f1)
        history["val_composite"].append(val_composite)

        if val_composite > best_val_composite:
            best_val_composite = val_composite
            best_val_macro_f1 = val_macro_f1
            torch.save(eval_net.state_dict(), ckpt_path)
            print(
                f"FT Epoch [{epoch+1:02d}/{ft_epochs}] | Val Macro-F1: {val_macro_f1*100:.2f}% | "
                f"Val Acc: {val_acc*100:.2f}% | Val Comp: {val_composite*100:.2f}% -> Lưu best mới!"
            )

    # --- GIAI ĐOẠN 3: TỐI ƯU NGƯỠNG HẬU NGHIỆM CÓ RÀNG BUỘC CHẶT ---
    model.freeze_backbone(False)
    model.load_state_dict(torch.load(ckpt_path, map_location=device, weights_only=True))
    model.eval()

    val_logits_list, val_targets_list = [], []
    with torch.no_grad():
        for images, soft_labels, reference_labels in val_loader:
            images = images.to(device)
            val_logits_list.append(model(images).cpu())
            val_targets_list.extend(reference_labels.cpu().numpy())

    val_logits_all = torch.cat(val_logits_list, dim=0)

    # C3: Tùy chọn log kết quả Val trước khi tối ưu vector tau
    if log_val_before_after_tau:
        pre_adj_val = apply_posthoc_logit_adjustment(val_logits_all, class_priors, tau=posthoc_tau)
        pre_preds = pre_adj_val.argmax(dim=1).numpy()
        pre_acc, pre_mf1, pre_wf1, pre_comp = compute_epoch_metrics(
            val_targets_list, pre_preds, alpha_composite
        )

    best_tau_vec = optimize_classwise_posthoc_tau(
        val_logits_all,
        val_targets_list,
        class_priors,
        base_tau=posthoc_tau,
        max_acc_drop=0.005,
        alpha_composite=alpha_composite,
        rare_classes=(5, 6, 7),
        max_rare_recall_drop=0.02,
        l2_reg=0.05,
    )

    adj_val_logits = apply_posthoc_logit_adjustment(val_logits_all, class_priors, tau=best_tau_vec)
    val_preds_cal = adj_val_logits.argmax(dim=1).numpy()
    cal_val_acc, cal_val_macro_f1, cal_val_wf1, cal_val_comp = compute_epoch_metrics(
        val_targets_list, val_preds_cal, alpha_composite
    )

    if log_val_before_after_tau:
        print(
            f"\n[LOG C3 | KIỂM TRA MỨC ĐỘ OVERFIT VAL KHI TỐI ƯU TAU - SEED {seed}]\n"
            f"  • Trước tối ưu (tau={posthoc_tau:.2f}) -> Val Acc: {pre_acc*100:.2f}% | "
            f"Val Macro-F1: {pre_mf1*100:.2f}% | Val Comp: {pre_comp*100:.2f}%\n"
            f"  • Sau tối ưu  (best_tau_vec) -> Val Acc: {cal_val_acc*100:.2f}% | "
            f"Val Macro-F1: {cal_val_macro_f1*100:.2f}% | Val Comp: {cal_val_comp*100:.2f}% "
            f"(Delta Comp: {(cal_val_comp - pre_comp)*100:+.2f}%)"
        )

    best_val_composite = cal_val_comp
    best_val_macro_f1 = cal_val_macro_f1

    # --- ĐÁNH GIÁ CUỐI CÙNG TRÊN TẬP TEST ---
    final_probabilities = predict_probabilities(model, X_test, y_test_hard, class_priors, best_tau_vec)
    y_true, y_pred = y_test_hard, final_probabilities.argmax(1)

    metrics = evaluate_private_test(
        y_true, y_pred, EMOTION_DICT, title_prefix=f"CustomCNN (Seed {seed} - Balanced LogitAdj)"
    )
    metrics["best_val_composite"] = best_val_composite
    metrics["best_val_macro_f1"] = best_val_macro_f1
    metrics["posthoc_tau_vec"] = best_tau_vec
    history_frame = pd.DataFrame(history)
    history_frame.insert(0, "epoch", np.arange(1, len(history_frame)+1))
    history_frame["phase"] = np.where(history_frame.epoch <= phase1_ran, "train", "fine_tune")
    history_frame.to_csv(RUN_DIR / f"history_seed{seed}.csv", index=False)
    for split, X, y in [("val", X_val, y_val_hard), ("test", X_test, y_test_hard)]:
        probabilities = predict_probabilities(model, X, y, class_priors, best_tau_vec)
        export_predictions(SPLIT_MANIFEST_DIR / f"{split}_ids.csv", y, probabilities,
                           RUN_DIR, f"seed{seed}_{split}_{EVAL_PROTOCOL}", LABEL_NAMES)
    metrics["seed"] = seed
    torch.save({"class_priors": class_priors.tolist(), "posthoc_tau": np.asarray(best_tau_vec).tolist()},
               RUN_DIR / f"calibration_seed{seed}.pt")
    return model, metrics, history, phase1_ran, ckpt_path


def predict_probabilities(model, X, y, priors, tau, protocol=None):
    protocol = protocol or EVAL_PROTOCOL
    ds = FER2013Dataset(X, y, is_train=False)
    model.eval()
    output = []
    if protocol == "single":
        with torch.no_grad():
            for images, _ in DataLoader(ds, batch_size=64, shuffle=False):
                logits = model(images.to(device))
                output.extend(apply_posthoc_logit_adjustment(logits, priors, tau=tau).softmax(1).cpu().numpy())
    elif protocol == "ten_crop":
        for image, _ in ds:
            _, probs = ten_crop_inference(model, image, pad_size=4, crop_size=48,
                                         class_priors=priors, posthoc_tau=tau)
            output.append(np.asarray(probs))
    else:
        raise ValueError(protocol)
    return np.asarray(output)


In [ ]:
# CELL 3: TRAIN BOTH BRANCHES; SELECT EACH BEST MODEL USING ITS OWN VALIDATION
for branch in BRANCHES:
    activate_branch(branch)
    custom_cnn_results = []
    best_custom_model = None
    best_val_score_across_seeds = -1.0
    best_tau_vector = None

    split_cfg_path = SPLIT_MANIFEST_DIR / "split_config.json"
    split_config_hash = (
        compute_file_sha256(split_cfg_path) if split_cfg_path.exists() else "no_manifest_saved"
    )

    for seed in SEEDS:
        model_run, metrics_run, hist_run, p1_epochs, ckpt = train_and_evaluate_custom_cnn(
            seed=seed,
            num_epochs=85,
            ft_epochs=15,
            patience=25,
            mixup_alpha=0.2,
            mixup_prob=0.4,
            stage1_tau=0.30,
            stage1_max_gamma=0.5,
            gamma_warmup_epochs=15,
            posthoc_tau=0.10,
            use_ema=True,
            alpha_composite=0.5,
            sampler_use_sample_weight=True,
            log_val_before_after_tau=False,
        )
        custom_cnn_results.append(metrics_run)

        # Chỉ dùng điểm trên tập Validation (best_val_composite) để chọn mô hình lưu best.pth & best_bundle.pth
        if metrics_run["best_val_composite"] > best_val_score_across_seeds:
            best_seed = seed
            best_val_score_across_seeds = metrics_run["best_val_composite"]
            best_custom_model = model_run
            best_tau_vector = metrics_run["posthoc_tau_vec"]

            # 1. Lưu best.pth và best_calibration.pt để giữ tương thích ngược
            torch.save(model_run.state_dict(), "best.pth")
            torch.save(
                {
                    "class_priors": np.asarray(class_priors, dtype=np.float32).tolist(),
                    "posthoc_tau": np.asarray(best_tau_vector, dtype=np.float32).tolist(),
                },
                "best_calibration.pt",
            )

            # 2. Lưu best_bundle.pth (chuyển toàn bộ numpy array sang list/tensor chuẩn để nạp bằng weights_only=True)
            bundle_dict = {
                "seed": seed,
                "experiment": EXPERIMENT,
                "evaluation_protocol": EVAL_PROTOCOL,
                "model_state_dict": model_run.state_dict(),
                "class_priors": np.asarray(class_priors, dtype=np.float32).tolist(),
                "posthoc_tau_vec": np.asarray(best_tau_vector, dtype=np.float32).tolist(),
                "use_eca": bool(model_run.block1.eca is not None),
                "use_spatial_attn": bool(model_run.block1.spatial_attn is not None),
                "label_names": list(LABEL_NAMES),
                "split_config_hash": str(split_config_hash),
                "preprocessing_config": dict(PREPROCESSING_CONFIG),
            }
            torch.save(bundle_dict, "best_bundle.pth")

            best_custom_history = (hist_run, p1_epochs)
            print(
                f"--> Đã cập nhật best.pth, best_calibration.pt & best_bundle.pth từ Seed {seed} "
                f"(Val Composite = {best_val_score_across_seeds*100:.2f}%)"
            )

    summary_custom_cnn = summarize_multi_seed_results(
        custom_cnn_results,
        model_name=f"CustomCNN ({EXPERIMENT}, {len(SEEDS)} seeds)",
    )
    plot_learning_curves(best_custom_history[0], phase1_epochs=best_custom_history[1])
    # Flatten scalar results for spreadsheet use; nested metadata is saved separately.
    scalar_rows = [{k: v for k,v in m.items() if np.isscalar(v)} for m in custom_cnn_results]
    metrics_df = pd.DataFrame(scalar_rows)
    metrics_df.to_csv(RUN_DIR / "metrics_per_seed.csv", index=False)
    metrics_df.drop(columns="seed").agg(["mean", "std"]).T.to_csv(RUN_DIR / "metrics_summary.csv", index_label="metric")

    del model_run, best_custom_model
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
os.chdir(EXPERIMENT_ROOT)


In [ ]:
# CELL 4: OWN-TEST RESULTS AND TRUE/PREDICTED GRAD-CAM FOR EACH BRANCH
import matplotlib.pyplot as plt
own_test_rows = []
for branch in BRANCHES:
    activate_branch(branch)
    bundle = torch.load(RUN_DIR / "best_bundle.pth", map_location="cpu", weights_only=True)
    if bundle["split_config_hash"] != compute_file_sha256(SPLIT_MANIFEST_DIR / "split_config.json"):
        raise ValueError("Checkpoint belongs to a different split; train this branch first.")
    if bundle["experiment"] != branch or bundle["evaluation_protocol"] != EVAL_PROTOCOL:
        raise ValueError("Checkpoint branch or evaluation protocol differs.")
    if bundle["preprocessing_config"] != PREPROCESSING_CONFIG or bundle["label_names"] != list(LABEL_NAMES):
        raise ValueError("Checkpoint preprocessing or class order differs.")
    best_custom_model = CustomCNN(num_classes=8, use_eca=bundle["use_eca"],
                                  use_spatial_attn=bundle["use_spatial_attn"]).to(device)
    best_custom_model.load_state_dict(bundle["model_state_dict"])
    class_priors = np.asarray(bundle["class_priors"])
    best_tau_vector = np.asarray(bundle["posthoc_tau_vec"])
    for split, X, y in [("val", X_val, y_val_hard), ("test", X_test, y_test_hard)]:
        manifest = pd.read_csv(SPLIT_MANIFEST_DIR / f"{split}_ids.csv")
        probs = predict_probabilities(best_custom_model, X, y, class_priors, best_tau_vector)
        frame = export_predictions(manifest, y, probs, RUN_DIR,
                                   f"selected_{split}_{EVAL_PROTOCOL}", LABEL_NAMES)
        if split == "test":
            metrics = evaluate_private_test(y, probs.argmax(1), EMOTION_DICT,
                                            title_prefix=f"{branch} — own test")
            own_test_rows.append(dict(branch=branch, n=len(y), protocol=EVAL_PROTOCOL,
                                     **{k: v for k, v in metrics.items() if np.isscalar(v)}))
            untied = ~frame.target_is_tie.to_numpy()
            if untied.any():
                export_predictions(manifest.loc[untied].reset_index(drop=True), y[untied], probs[untied],
                                   RUN_DIR, f"selected_test_untied_{EVAL_PROTOCOL}", LABEL_NAMES)

    # Grad-CAM: single-crop explanation for the calibrated predicted class.
    # When evaluation uses ten crops, this panel is explicitly a single-crop diagnostic.
    test_ds_vis = FER2013Dataset(X_test, y_test_hard, is_train=False)
    vis_probs = predict_probabilities(best_custom_model, X_test, y_test_hard,
                                     class_priors, best_tau_vector, protocol="single")
    fig, axes = plt.subplots(2, 8, figsize=(24, 8))
    cam_rows = []
    test_manifest = pd.read_csv(SPLIT_MANIFEST_DIR / "test_ids.csv")
    for cls_idx in range(8):
        positions = np.where(y_test_hard == cls_idx)[0]
        if not len(positions):
            axes[0, cls_idx].axis("off"); axes[1, cls_idx].axis("off")
            continue
        # Prefer an error when available to make the diagnostic useful.
        errors = positions[vis_probs[positions].argmax(1) != cls_idx]
        pos = int(errors[0] if len(errors) else positions[0])
        img_tensor, _ = test_ds_vis[pos]
        pred_cls = int(vis_probs[pos].argmax())
        confidence = float(vis_probs[pos, pred_cls])
        true_classes = np.flatnonzero(y_test_soft[pos] == y_test_soft[pos].max())
        true_text = " / ".join(LABEL_NAMES[c] for c in true_classes)
        correct = pred_cls == int(y_test_hard[pos])
        matches_any_top = pred_cls in true_classes
        cam, _ = generate_gradcam(best_custom_model, img_tensor, target_class=pred_cls)
        orig = denormalize_image(img_tensor)
        axes[0, cls_idx].imshow(orig, cmap="gray")
        axes[0, cls_idx].set_title(f"Thật: {true_text}\nTham chiếu: {LABEL_NAMES[int(y_test_hard[pos])]}\nDự đoán: {LABEL_NAMES[pred_cls]}\n{confidence:.1%} | {'ĐÚNG' if correct else 'SAI'}",
                                   color="green" if correct else "red", fontsize=10)
        axes[1, cls_idx].imshow(orig, cmap="gray")
        axes[1, cls_idx].imshow(cam, cmap="jet", alpha=.45)
        axes[1, cls_idx].set_title(f"Grad-CAM: {LABEL_NAMES[pred_cls]}", fontsize=10)
        for row in range(2): axes[row, cls_idx].axis("off")
        cam_rows.append(dict(image_id=test_manifest.iloc[pos].image_id, true_label=true_text,
                             predicted_label=LABEL_NAMES[pred_cls], confidence=confidence, correct=correct, reference_label=LABEL_NAMES[int(y_test_hard[pos])],
                             matches_any_top_class=matches_any_top, protocol="single", cam_target=LABEL_NAMES[pred_cls]))
    fig.suptitle("Grad-CAM — các lớp nhãn thật đồng hạng; ĐÚNG/SAI theo nhãn tham chiếu cố định")
    fig.tight_layout()
    fig.savefig(RUN_DIR / "gradcam_true_vs_predicted.png", dpi=180, bbox_inches="tight")
    plt.show()
    plt.close(fig)
    pd.DataFrame(cam_rows).to_csv(RUN_DIR / "gradcam_predictions.csv", index=False)


    # Bind predictions to these selected weights and this manifest (reject stale files in cell 5).
    (RUN_DIR / "evaluation_metadata.json").write_text(json.dumps({
        "branch": branch, "protocol": EVAL_PROTOCOL,
        "bundle_sha256": compute_file_sha256(RUN_DIR / "best_bundle.pth"),
        "test_manifest_sha256": compute_file_sha256(SPLIT_MANIFEST_DIR / "test_ids.csv"),
        "predictions_sha256": compute_file_sha256(RUN_DIR / f"predictions_selected_test_{EVAL_PROTOCOL}.csv"),
    }, indent=2), encoding="utf-8")
    del best_custom_model, bundle
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

os.chdir(EXPERIMENT_ROOT)
own_test_summary = pd.DataFrame(own_test_rows)
own_test_summary.to_csv(EXPERIMENT_ROOT / "comparison_own_tests.csv", index=False)
display(own_test_summary)
print("These rows use DIFFERENT tests; McNemar uses only the intersection in cell 5.")


In [ ]:
# CELL 5: EXACT McNEMAR ON THE TEST INTERSECTION — NO TRAINING OR RESPLITTING
comparison_dir = EXPERIMENT_ROOT / "test_intersection"
result = compare_test_intersection(
    EXPERIMENT_ROOT / "drop_ties", EXPERIMENT_ROOT / "keep_ties",
    comparison_dir, LABEL_NAMES, protocol=EVAL_PROTOCOL,
)
display(result)
print("Old/baseline = drop_ties; new/candidate = keep_ties.")
print("McNemar tests paired correctness, not Macro-F1. The intersection contains untied examples only.")
print("Its class distribution is reported separately; it is not guaranteed stratified.")

# Preserve all per-branch CSV exports plus intersection comparison in one download.
import shutil
archive = shutil.make_archive(str(EXPERIMENT_ROOT.parent / (EXPERIMENT_ROOT.name + "_results")),
                              "zip", EXPERIMENT_ROOT)
print("Results:", archive)
